In [66]:
import torch
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("../")
DATASET_PATH = PROJECT_ROOT / "datasets" / "depth_v1"

In [67]:
from torch.utils.data import Dataset

class DepthHandDataset(Dataset):
    def __init__(self, npz_paths, frames_per_clip=60):
        self.npz_paths = list(npz_paths)
        self.frames_per_clip = frames_per_clip
        self.total_frames = len(self.npz_paths) * self.frames_per_clip
        
    def __len__(self):
        return self.total_frames
    
    def __getitem__(self, idx):
        clip_idx = idx // self.frames_per_clip
        frame_idx = idx % self.frames_per_clip
        
        clip_path = self.npz_paths[clip_idx]
        
        with np.load(clip_path) as data:
            depth = data["depth"][frame_idx]
            joints = data["joints_3d"][frame_idx]
            
            x = torch.from_numpy(depth).float().unsqueeze(0)
            y = torch.from_numpy(joints).float() 
            return x, y

In [68]:
import random
from torch.utils.data import DataLoader
SPLIT_RATIO = 0.9

random.seed(42)
npz_clips = sorted(list(DATASET_PATH.glob("*.npz")))
clips_shuffled = npz_clips.copy()
random.shuffle(clips_shuffled)

split_idx = int(SPLIT_RATIO * len(clips_shuffled))
train_files = clips_shuffled[:split_idx]
val_files = clips_shuffled[split_idx:]

print(f"Clips for TRAIN: {len(train_files)} ({len(train_files)*60} frames)")
print(f"Clips for VAL:   {len(val_files)} ({len(val_files)*60} frames)")

train_dataset = DepthHandDataset(train_files)
val_dataset = DepthHandDataset(val_files)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Clips for TRAIN: 425 (25500 frames)
Clips for VAL:   48 (2880 frames)


In [69]:
import torch.nn as nn
import torchvision.models as models

class HandPoseResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.backbone = models.resnet18(weights=None)
        old_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,                      # as simple resnet, but with 1-channel input
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, 21 * 3)

    def forward(self, x):
        out = self.backbone(x)           # (Batch, 63)
        return out.view(-1, 21, 3)       # (Batch, 21, 3)

In [70]:
net = HandPoseResNet18()

In [71]:
def mpjpe(y_hat, y_gt):
    diff = y_hat - y_gt # (Batch, 21, 3)
    dist = torch.linalg.vector_norm(diff, dim=-1) 
    
    return dist.mean() * 1000 # mjpe in mm
 
criterion = nn.MSELoss()

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = HandPoseResNet18().to(device)

In [73]:
from tqdm.auto import tqdm

EPOCHS = 10

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val_mpjpe = float("inf")

for epoch in range(EPOCHS):
    # TRAIN
    net.train()
    train_loss_total = 0.0
    train_mpjpe_total = 0.0

    train_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for x, y in train_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        y_hat = net(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item()
        with torch.no_grad():
            batch_mpjpe = mpjpe(y_hat, y).item()
            train_mpjpe_total += batch_mpjpe

        train_bar.set_postfix({
            "loss": f"{loss.item():.5f}",
            "mpjpe": f"{batch_mpjpe:.1f} mm"
        })

    train_loss_avg = train_loss_total / len(train_dataloader)
    train_mpjpe_avg = train_mpjpe_total / len(train_dataloader)

    # VAL
    net.eval()
    val_loss_total = 0.0
    val_mpjpe_total = 0.0

    val_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_hat = net(x)
            loss = criterion(y_hat, y)

            val_loss_total += loss.item()
            val_mpjpe_total += mpjpe(y_hat, y).item()

    val_loss_avg = val_loss_total / len(val_dataloader)
    val_mpjpe_avg = val_mpjpe_total / len(val_dataloader)

    scheduler.step()

    # EPOCH SUMMARY
    print(f"\n>>> Epoch {epoch+1}")
    print(f"Train Loss: {train_loss_avg:.5f} | Train MPJPE: {train_mpjpe_avg:.2f} mm")
    print(f"Val Loss:   {val_loss_avg:.5f} | Val MPJPE:   {val_mpjpe_avg:.2f} mm")
    print(f"Current LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_mpjpe_avg < best_val_mpjpe:
        best_val_mpjpe = val_mpjpe_avg
        torch.save(net.state_dict(), "best_hand_model.pth")
        print(f"New best model saved! (MPJPE: {best_val_mpjpe:.2f} mm)")
    print("\n")

Epoch 1/10 [Train]:   0%|          | 0/797 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/90 [00:00<?, ?it/s]


>>> Epoch 1
Train Loss: 0.00073 | Train MPJPE: 20.80 mm
Val Loss:   0.00009 | Val MPJPE:   13.75 mm
Current LR: 9.76e-05
 New best model saved! (MPJPE: 13.75 mm)




Epoch 2/10 [Train]:   0%|          | 0/797 [00:00<?, ?it/s]

Epoch 2/10 [Val]:   0%|          | 0/90 [00:00<?, ?it/s]

...

Epoch 10/10 [Train]:   0%|          | 0/797 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x748c5d54f240>
Traceback (most recent call last):
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x748c5d54f240>
Traceback (most recent call last):
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers(

Epoch 10/10 [Val]:   0%|          | 0/90 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x748c5d54f240>
Traceback (most recent call last):
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x748c5d54f240>
Traceback (most recent call last):
  File "/home/lashin_ma/.conda/envs/ffs/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers(


>>> Epoch 10
Train Loss: 0.00001 | Train MPJPE: 4.66 mm
Val Loss:   0.00002 | Val MPJPE:   5.61 mm
Current LR: 0.00e+00
 New best model saved! (MPJPE: 5.61 mm)


